# Biblical Mistral-Nemo 13B Fine-Tuning with Unsloth (4-bit QLoRA)

**Base Model:** Mistral-Nemo 13B Instruct

**Dataset:** Augmentoolkit-generated Biblical persona (first-person apostolic responses)

**Training Hardware:** NVIDIA DGX Spark (128GB unified memory)

**Deployment Target:** A5000 GPU server via vLLM (24GB VRAM — 13B 4-bit ~8GB fits comfortably)

**Chat Template:** `[INST] user message [/INST] assistant response</s>`

## Step 1: Configuration

All paths and variables for easy configuration.

In [ ]:
# ============================================================================
# CONFIGURATION - All variables for easy setup
# ============================================================================
# Training: DGX Spark (128GB unified memory)
# Deployment: A5000 GPU server (24GB VRAM) via vLLM (13B 4-bit ~8GB)

# =========================== MODEL CONFIGURATION ===========================
# Base model to fine-tune (Unsloth-optimized)
BASE_LLM = "unsloth/Mistral-Nemo-Instruct-2407"

# Name for output folders and model identification
MODEL_NAME_BASE = "biblical_mistral_nemo_13b_instruct_unsloth"

# =========================== INPUT DATA ===========================
# Path to training data (Augmentoolkit output)
INPUT_DATA_PATH = "/home/spark/projects/augmentoolkit/outputs/biblical_persona_dataset"

# =========================== OUTPUT DIRECTORIES ===========================
# All outputs organized under ./output/{MODEL_NAME_BASE}/
OUTPUT_BASE_DIR = f"./output/{MODEL_NAME_BASE}"
OUTPUT_DIR_ADAPTERS = f"{OUTPUT_BASE_DIR}/train"       # LoRA weights during training

# =========================== TRAINING HYPERPARAMETERS ===========================
MAX_SEQ_LENGTH = 2048    # Max tokens per example
BATCH_SIZE = 2           # Per-device batch size
GRAD_ACCUM = 8           # Gradient accumulation (effective batch = 16)
LEARNING_RATE = 2e-5     # Learning rate
TARGET_EPOCHS = 1        # Number of training epochs

# =========================== LoRA CONFIGURATION ===========================
# LoRA (Low-Rank Adaptation) allows efficient fine-tuning by only training
# small adapter matrices instead of full model weights.
#
# LORA_R (Rank): Controls adapter capacity
#   - Lower (4-8): Less aggressive, preserves base model behavior
#   - Higher (16-64): More capacity for new knowledge, risk of overfitting
#   - Recommendation: 8 for 13B QLoRA (model is already very capable)
#
# LORA_ALPHA: Scaling factor for LoRA weights
#   - Typically set to 2x LORA_R (e.g., r=8 → alpha=16)
#   - Higher alpha = stronger influence of fine-tuning
#
# LORA_DROPOUT: Regularization during training
#   - 0: No dropout (faster training)
#   - 0.05-0.1: Mild regularization for larger datasets
#
LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0

# =========================== LoRA TARGET MODULES ===========================
# Which layers to fine-tune. Mistral architecture has these trainable layers:
#
# ATTENTION MODULES (Recommended for persona/style transfer)
# ┌──────────────┬─────────────────────────┬─────────────────────────────────────┐
# │ Module       │ Function                │ Training Impact                     │
# ├──────────────┼─────────────────────────┼─────────────────────────────────────┤
# │ q_proj       │ Query projection        │ "What am I looking for?"            │
# │              │                         │ Core attention steering             │
# ├──────────────┼─────────────────────────┼─────────────────────────────────────┤
# │ k_proj       │ Key projection          │ "What info do I have?"              │
# │              │                         │ Memory/context matching             │
# ├──────────────┼─────────────────────────┼─────────────────────────────────────┤
# │ v_proj       │ Value projection        │ "What info to pass forward?"        │
# │              │                         │ Content selection                   │
# ├──────────────┼─────────────────────────┼─────────────────────────────────────┤
# │ o_proj       │ Output projection       │ Combines attention heads            │
# │              │                         │ Post-attention mixing               │
# └──────────────┴─────────────────────────┴─────────────────────────────────────┘
#
# MLP/FFN MODULES (More aggressive - use for knowledge injection)
# ┌──────────────┬─────────────────────────┬─────────────────────────────────────┐
# │ Module       │ Function                │ Training Impact                     │
# ├──────────────┼─────────────────────────┼─────────────────────────────────────┤
# │ gate_proj    │ Gate projection         │ Controls information flow           │
# │              │                         │ Heavy knowledge modification        │
# ├──────────────┼─────────────────────────┼─────────────────────────────────────┤
# │ up_proj      │ Up projection           │ Expands to hidden dimension         │
# │              │                         │ Feature extraction                  │
# ├──────────────┼─────────────────────────┼─────────────────────────────────────┤
# │ down_proj    │ Down projection         │ Compresses back to model dim        │
# │              │                         │ Output generation                   │
# └──────────────┴─────────────────────────┴─────────────────────────────────────┘
#
# PRESETS:
#   Conservative (style/persona):  ["q_proj", "v_proj"]
#   Balanced:                      ["q_proj", "k_proj", "v_proj", "o_proj"]
#   Aggressive (new knowledge):    ["q_proj", "k_proj", "v_proj", "o_proj", 
#                                   "gate_proj", "up_proj", "down_proj"]
#
LORA_TARGET_MODULES = ["q_proj", "v_proj"]  # Conservative: attention-only

# =========================== VLLM DEPLOYMENT ===========================
# Centralized vLLM models directory on A5000 server (mapped to /models in container)
VLLM_MODELS_DIR = "/home/spark/projects/stoic/output/vllm"

# =========================== INFERENCE TEST ===========================
# Test prompt for inference validation
TEST_PROMPT = "I am struggling with forgiveness. What does Scripture teach about forgiving others?"

# ============================================================================
print("✓ Configuration loaded (13B 4-bit QLoRA version)")
print(f"  Base model: {BASE_LLM}")
print(f"  Model name: {MODEL_NAME_BASE}")
print(f"  Input data: {INPUT_DATA_PATH}")
print(f"  Output base: {OUTPUT_BASE_DIR}")
print(f"  LoRA config: r={LORA_R}, alpha={LORA_ALPHA}, targets={LORA_TARGET_MODULES}")
print(f"  Training: batch={BATCH_SIZE}, grad_accum={GRAD_ACCUM}, lr={LEARNING_RATE}")
print(f"  Training precision: 4-bit QLoRA")
print(f"  Deployment: vLLM on A5000 GPU server")

## 2. Environment Preparation

Install Unsloth and updated HuggingFace libraries.

In [ ]:
# Install core packages from PyPI (much faster than git installs)
!pip install -q unsloth transformers trl peft accelerate datasets bitsandbytes

# Verify installations
import unsloth
import transformers
import trl
print(f"✓ Unsloth: {unsloth.__version__}")
print(f"✓ Transformers: {transformers.__version__}")
print(f"✓ TRL: {trl.__version__}")
print("Environment ready!")

## 3. Load Dataset & Format for Instruction Tuning

Load the Augmentoolkit-generated Biblical dataset (first-person apostolic responses from Paul, David, Peter, etc.).

In [ ]:
from datasets import load_dataset, concatenate_datasets
import glob

# Load ALL subdirectories and ALL files
all_dirs = glob.glob(f"{INPUT_DATA_PATH}/*/")

print("📚 LOADING ALL AUGMENTOOLKIT DATA")
print(f"Found {len(all_dirs)} subdirectories")

datasets = []
for dir_path in sorted(all_dirs):
    jsonl_files = glob.glob(f"{dir_path}/*.jsonl")
    for file_path in jsonl_files:
        try:
            ds = load_dataset("json", data_files=file_path, split="train")
            datasets.append(ds)
            print(f"  Loaded {len(ds)} from {dir_path.split('/')[-2]}/{file_path.split('/')[-1]}")
        except Exception as e:
            print(f"  Skipped {file_path.split('/')[-1]}: {e}")

dataset = concatenate_datasets(datasets)

print(f"\n✓ Total dataset loaded: {len(dataset)} examples")
print(f"✓ Columns: {dataset.column_names}")
print(f"\n--- First Example (first 800 chars) ---")
import json
print(json.dumps(dataset[0], indent=2)[:800])

# Shuffle dataset for better training
dataset = dataset.shuffle(seed=42)
print(f"\n✓ Dataset shuffled and ready for training")

## 4. Load Model & Tokenizer with Unsloth

Load Mistral-Nemo 13B Instruct model with 4-bit quantization for training efficiency.

In [ ]:
from unsloth import FastLanguageModel
import torch

# Load model with 4-bit quantization for training efficiency
model, tokenizer = FastLanguageModel.from_pretrained(
    BASE_LLM,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,           # Auto-detect best dtype
    load_in_4bit=True,    # 4-bit quantization for 13B model
    device_map={"": 0}    # Force all on GPU 0
)

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"✓ Model loaded: {BASE_LLM}")
print(f"✓ Precision: 4-bit quantized (NF4)")
print(f"✓ Tokenizer configured")
print(f"✓ Max sequence length: {MAX_SEQ_LENGTH}")

In [ ]:
# Format dataset for Mistral chat template
# Handle BOTH ShareGPT format (conversations) AND raw text format
def format_instruct(example):
    # If has conversations field AND it's not null, convert ShareGPT to chat template
    if example.get("conversations") is not None:
        messages = []
        for turn in example["conversations"]:
            # ShareGPT uses "from": "system"/"human"/"gpt"
            # Standard uses "role": "system"/"user"/"assistant"
            if turn["from"] == "system":
                messages.append({"role": "system", "content": turn["value"]})
            elif turn["from"] == "human":
                messages.append({"role": "user", "content": turn["value"]})
            elif turn["from"] == "gpt":
                messages.append({"role": "assistant", "content": turn["value"]})
        
        text = tokenizer.apply_chat_template(
            messages, 
            tokenize=False, 
            add_generation_prompt=False
        )
        return {"text": text}
    
    # Otherwise if it has a text field (raw text files), keep it as-is
    elif example.get("text") is not None and len(str(example["text"])) > 0:
        return {"text": str(example["text"])}
    
    # Skip malformed examples
    return {"text": ""}

# Format and keep only text column
dataset = dataset.map(format_instruct, remove_columns=dataset.column_names)

# Filter out empty examples
dataset = dataset.filter(lambda x: len(x["text"]) > 0)

print(f"\n--- Sample formatted text (first 500 chars) ---")
print(dataset[0]['text'][:500])
print(f"\n✓ Dataset formatted: {len(dataset)} examples")

## 5. Add LoRA Adapters

Configure LoRA for efficient fine-tuning. See configuration section for module explanations.

In [ ]:
from peft import LoraConfig

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=LORA_TARGET_MODULES,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    max_seq_length=MAX_SEQ_LENGTH
)

print(f"✓ LoRA adapters added (r={LORA_R}, targets={LORA_TARGET_MODULES})")
print(f"✓ Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## 6. Trainer Setup & Training

**PURE DOMAIN DATA with LIGHT TRAINING:**
- 100% authentic Biblical examples from epistles and psalms
- 1 epoch with low learning rate (2e-5) to gently teach persona
- 13B is already highly capable — minimal fine-tuning is sufficient
- This preserves base model capabilities while adding authentic voice

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

# Calculate training steps
effective_batch_size = BATCH_SIZE * GRAD_ACCUM
steps_per_epoch = len(dataset) // effective_batch_size
max_steps = steps_per_epoch * TARGET_EPOCHS
warmup_steps = max(1, max_steps // 10)
save_steps = min(200, steps_per_epoch)  # Save every 200 steps or at epoch end

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        warmup_steps=warmup_steps,
        max_steps=max_steps,
        learning_rate=LEARNING_RATE,
        fp16=True,
        bf16=False,
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=3407,
        output_dir=OUTPUT_DIR_ADAPTERS,
        report_to="none",
        save_strategy="steps",
        save_steps=save_steps,
    )
)

print("✓ Trainer configured for PURE DOMAIN (light training)")
print(f"✓ Dataset size: {len(dataset)} conversations")
print(f"✓ Effective batch size: {effective_batch_size} (batch={BATCH_SIZE} × grad_accum={GRAD_ACCUM})")
print(f"✓ Steps per epoch: {steps_per_epoch}")
print(f"✓ Total epochs: {TARGET_EPOCHS}")
print(f"✓ Total steps: {max_steps}")
print(f"✓ Warmup steps: {warmup_steps}")
print(f"✓ Save every: {save_steps} steps (every epoch)")

In [ ]:
# Start training
trainer.train()

## 7. Save LoRA Adapters

Save the trained LoRA adapters separately. These can be loaded on any Mistral-Nemo 13B model later.

**This is the primary output** - merging to full model is optional (Step 8).

In [ ]:
from pathlib import Path

# Save LoRA adapters (primary output)
LORA_OUTPUT_DIR = f"{OUTPUT_BASE_DIR}/lora_adapters"
Path(LORA_OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

print(f"💾 Saving LoRA adapters to {LORA_OUTPUT_DIR}...")
model.save_pretrained(LORA_OUTPUT_DIR)
tokenizer.save_pretrained(LORA_OUTPUT_DIR)

print(f"✅ LoRA adapters saved!")
print(f"   Path: {LORA_OUTPUT_DIR}")
print(f"   Can be loaded on any Mistral-Nemo 13B model with PEFT")